In [5]:
from __future__ import annotations



from pathlib import Path

import json

import random



import numpy as np

import pandas as pd

import geopandas as gpd

from shapely.geometry import box, mapping



# ---- Satellite querying (Planetary Computer STAC) ----

# Make this cell robust to missing packages inside the *notebook kernel*.

try:

    from pystac_client import Client

    import planetary_computer as pc

except ModuleNotFoundError:

    import sys

    import subprocess

    print('Installing missing satellite deps into kernel...')

    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pystac-client', 'planetary-computer'])

    from pystac_client import Client

    import planetary_computer as pc



import rasterio

import rasterio.mask

from rasterio.vrt import WarpedVRT



import matplotlib

matplotlib.use('Agg')  # never inline images

import matplotlib.pyplot as plt





def _find_repo_root(start: Path) -> Path:

    start = start.resolve()

    for p in [start] + list(start.parents):

        # Heuristic: repo root should contain output/ and src/

        if (p / 'output').exists() and (p / 'src').exists():

            return p

        if (p / 'Readme.md').exists() or (p / 'README.md').exists():

            return p

    return start





REPO_ROOT = _find_repo_root(Path.cwd())

OUTPUT_DIR = REPO_ROOT / 'output'

print('REPO_ROOT:', REPO_ROOT)

print('OUTPUT_DIR:', OUTPUT_DIR)





# ---- User-configurable parameters ----

DATASET_NAME = 'sit'  # just a label for outputs



YEAR = 2024  # change as needed

START_DATE = f'{YEAR}-01-01'

END_DATE   = f'{YEAR}-12-31'



# If None, we will auto-pick a likely crowns file from output/

CROWNS_PATH: str | None = None



# Number of crowns to compute NDVI TS for (kept small for speed)

N_EXAMPLE_CROWNS = 5

RANDOM_SEED = 7



# Cloud filter for Sentinel-2 items

MAX_CLOUD_PCT = 30



# Where to write plots/CSVs produced by this notebook

RUN_DIR = OUTPUT_DIR / 'sat_data' / f'{DATASET_NAME}_{YEAR}_run'

RUN_DIR.mkdir(parents=True, exist_ok=True)

print('RUN_DIR:', RUN_DIR.resolve())


REPO_ROOT: /Users/hbot07/VS Code/Drone-Phenology-Monitoring
OUTPUT_DIR: /Users/hbot07/VS Code/Drone-Phenology-Monitoring/output
RUN_DIR: /Users/hbot07/VS Code/Drone-Phenology-Monitoring/output/sat_data/sit_2024_run


In [14]:
def _pick_default_crowns_path(output_dir: Path) -> Path:

    # If we're working with the SIT dataset, prefer the SIT tracking run's crowns.

    # This matches the crown set used by the interactive SIT viewer (in georeferenced coords).

    if DATASET_NAME.lower() == 'sit':

        sit_geojson = output_dir / 'sit_tracking_rerun_1Apr26' / 'consensus_crowns_om1_phenology_sit.geojson'

        if sit_geojson.exists():

            return sit_geojson



    # Preference order: tree-only / phenology-ready / consensus crowns.

    candidates = [

        output_dir / 'consensus_crowns_tree_only.gpkg',

        output_dir / 'consensus_crowns_complete_all_tree_only.gpkg',

        output_dir / 'consensus_crowns_complete_all.gpkg',

        output_dir / 'consensus_crowns_tree_only_hard_rule.gpkg',

        output_dir / 'consensus_crowns_om1_phenology.geojson',

        output_dir / 'consensus_crowns_all_chains.gpkg',

    ]



    for p in candidates:

        if p.exists():

            return p



    # Last resort: any file matching consensus crowns

    for p in sorted(output_dir.glob('consensus_crowns*.*')):

        if p.suffix.lower() in {'.gpkg', '.geojson', '.json'}:

            return p



    raise FileNotFoundError(

        f'Could not auto-find a crowns file under {output_dir}. Set CROWNS_PATH explicitly.'

    )





crowns_path = Path(CROWNS_PATH) if CROWNS_PATH else _pick_default_crowns_path(OUTPUT_DIR)

print('Using crowns file:', crowns_path)



# If GPKG, we may need to choose a layer; try common ones and fall back to first.

crowns: gpd.GeoDataFrame

if crowns_path.suffix.lower() == '.gpkg':

    layers = list(gpd.list_layers(crowns_path).name)

    preferred = [

        'crowns',

        'consensus_crowns',

        'tree_only',

        'crowns_tree_only',

    ]

    layer = next((ly for ly in preferred if ly in layers), layers[0] if layers else None)

    if layer is None:

        raise ValueError(f'No layers found in {crowns_path}')

    print('Using layer:', layer)

    crowns = gpd.read_file(crowns_path, layer=layer)

else:

    crowns = gpd.read_file(crowns_path)



crowns = crowns[crowns.geometry.notnull() & ~crowns.geometry.is_empty].copy()

crowns = crowns.reset_index(drop=True)

print('n_crowns:', len(crowns))

print('crowns.crs:', crowns.crs)



bounds = crowns.total_bounds  # [minx, miny, maxx, maxy]

print('crowns.bounds:', bounds)



# Build an AOI bbox and also a WGS84 bbox for STAC search

aoi_poly = box(*bounds)

aoi = gpd.GeoDataFrame({'name': ['aoi']}, geometry=[aoi_poly], crs=crowns.crs)

aoi_wgs84 = aoi.to_crs('EPSG:4326')

minx, miny, maxx, maxy = aoi_wgs84.total_bounds

bbox_wgs84 = [float(minx), float(miny), float(maxx), float(maxy)]



centroid_wgs84 = aoi_wgs84.geometry.iloc[0].centroid

print('AOI centroid (lat, lon):', float(centroid_wgs84.y), float(centroid_wgs84.x))

print('AOI bbox WGS84:', bbox_wgs84)



# Persist a small metadata file for reproducibility

meta = {

    'dataset': DATASET_NAME,

    'year': YEAR,

    'start_date': START_DATE,

    'end_date': END_DATE,

    'crowns_path': str(crowns_path),

    'crowns_crs': str(crowns.crs),

    'aoi_bbox_wgs84': bbox_wgs84,

    'n_crowns': int(len(crowns)),

}

(RUN_DIR / 'run_meta.json').write_text(json.dumps(meta, indent=2), encoding='utf-8')

print('Wrote:', (RUN_DIR / 'run_meta.json'))


Using crowns file: /Users/hbot07/VS Code/Drone-Phenology-Monitoring/output/sit_tracking_rerun_1Apr26/consensus_crowns_om1_phenology_sit.geojson
n_crowns: 131
crowns.crs: EPSG:32643
crowns.bounds: [ 714282.90172129 3159477.74365683  714516.65620905 3159694.75926381]
AOI centroid (lat, lon): 28.54544367396415 77.19141903694393
AOI bbox WGS84: [77.19020487434051, 28.544445659547318, 77.19263322195405, 28.54644168390476]
Wrote: /Users/hbot07/VS Code/Drone-Phenology-Monitoring/output/sat_data/sit_2024_run/run_meta.json


In [15]:
# ---- 1) Query Sentinel-2 L2A items for the year over our AOI ----



import re



STAC_API = 'https://planetarycomputer.microsoft.com/api/stac/v1'

COLLECTION = 'sentinel-2-l2a'



catalog = Client.open(STAC_API)



search = catalog.search(

    collections=[COLLECTION],

    bbox=bbox_wgs84,

    datetime=f'{START_DATE}/{END_DATE}',

    query={'eo:cloud_cover': {'lt': MAX_CLOUD_PCT}},

)



items = list(search.get_items())

print('items_found:', len(items))





def _item_dt_str(it):

    dt = it.datetime

    return dt.strftime('%Y-%m-%d') if dt else None





def _sentinel_epsg_from_item(it, fallback_lat: float) -> int:

    # Prefer explicit proj:epsg if present

    epsg = it.properties.get('proj:epsg')

    if epsg is not None:

        return int(epsg)



    # Derive from MGRS tile / item id. Example tile: '43RGM' or id contains '_T43RGM_'.

    tile = it.properties.get('mgrs:tile')

    zone = None

    if isinstance(tile, str) and len(tile) >= 2 and tile[:2].isdigit():

        zone = int(tile[:2])

    if zone is None:

        m = re.search(r'_T(\d{2})[A-Z]{3}_', it.id)

        if m:

            zone = int(m.group(1))

    if zone is None:

        raise RuntimeError('Could not derive UTM zone for Sentinel item: ' + it.id)



    # Hemisphere from AOI latitude

    hemi_north = fallback_lat >= 0

    return (32600 + zone) if hemi_north else (32700 + zone)





rows = []

fallback_lat = float(centroid_wgs84.y)

for it in items:

    rows.append({

        'id': it.id,

        'datetime': _item_dt_str(it),

        'cloud_cover': it.properties.get('eo:cloud_cover'),

        'mgrs:tile': it.properties.get('mgrs:tile'),

        'sat:orbit_state': it.properties.get('sat:orbit_state'),

        'derived_epsg': _sentinel_epsg_from_item(it, fallback_lat=fallback_lat),

    })

items_df = pd.DataFrame(rows).sort_values(['cloud_cover', 'datetime']).reset_index(drop=True)



items_csv = RUN_DIR / 'sentinel2_items.csv'

items_df.to_csv(items_csv, index=False)

print('Wrote:', items_csv)



if items_df.empty:

    raise RuntimeError(

        'No Sentinel-2 items found for that year/AOI with the current cloud filter. Try increasing MAX_CLOUD_PCT.'

    )



# Pick the clearest item for visualization

best_item_id = items_df.iloc[0]['id']

best_item = next(it for it in items if it.id == best_item_id)

best_epsg = int(items_df.iloc[0]['derived_epsg'])

print('best_item:', best_item.id, 'date:', _item_dt_str(best_item), 'cloud:', best_item.properties.get('eo:cloud_cover'))

print('best_item derived EPSG:', best_epsg)


/Users/hbot07/anaconda3/envs/detectree/lib/python3.10/site-packages/pystac_client/item_search.py:925: FutureWarning: get_items() is deprecated, use items() instead
  warnings.warn(


items_found: 37
Wrote: /Users/hbot07/VS Code/Drone-Phenology-Monitoring/output/sat_data/sit_2024_run/sentinel2_items.csv
best_item: S2B_MSIL2A_20240518T052649_R105_T43RGM_20240518T091102 date: 2024-05-18 cloud: 3e-06
best_item derived EPSG: 32643


In [16]:
# ---- 2) Visualization: RGB underlay + crown outlines (saved to disk) ----



from pyproj import Transformer





def _signed_href(item, asset_key: str) -> str:

    asset = item.assets.get(asset_key)

    if asset is None:

        raise KeyError(f'Missing asset {asset_key!r} on item {item.id}')

    return pc.sign(asset.href)





def _read_window_resampled(src: rasterio.io.DatasetReader, window, max_dim: int = 1400):

    w = int(window.width)

    h = int(window.height)

    if w <= 0 or h <= 0:

        raise ValueError('Empty window')

    scale = max(w / max_dim, h / max_dim, 1.0)

    out_w = max(1, int(round(w / scale)))

    out_h = max(1, int(round(h / scale)))

    data = src.read(

        1,

        window=window,

        out_shape=(out_h, out_w),

        resampling=rasterio.enums.Resampling.bilinear,

    )

    transform = rasterio.windows.transform(window, src.transform)

    # Adjust transform for resampling

    transform = transform * rasterio.Affine.scale(w / out_w, h / out_h)

    return data, transform





def _percentile_stretch(x: np.ndarray, lo=2, hi=98):

    x = x.astype('float32')

    finite = np.isfinite(x)

    if not finite.any():

        return np.zeros_like(x, dtype='float32')

    vmin, vmax = np.percentile(x[finite], [lo, hi])

    if vmax <= vmin:

        return np.clip(x, 0, 1)

    y = (x - vmin) / (vmax - vmin)

    return np.clip(y, 0, 1)





# Sentinel-2 10m True Color: B04 (red), B03 (green), B02 (blue)

href_r = _signed_href(best_item, 'B04')

href_g = _signed_href(best_item, 'B03')

href_b = _signed_href(best_item, 'B02')



# IMPORTANT: rasterio CRS reprojection utilities are broken in this env due to a PROJ db mismatch.

# Use STAC-derived EPSG + pyproj for all coordinate transforms.

s2_epsg = _sentinel_epsg_from_item(best_item, fallback_lat=float(centroid_wgs84.y))

s2_crs_str = f'EPSG:{int(s2_epsg)}'



# Transform AOI bbox from WGS84 -> Sentinel CRS (corners are sufficient for small AOI)

minlon, minlat, maxlon, maxlat = bbox_wgs84

tx = Transformer.from_crs('EPSG:4326', s2_crs_str, always_xy=True)

xs, ys = tx.transform([minlon, minlon, maxlon, maxlon], [minlat, maxlat, minlat, maxlat])

aoi_bounds_s2 = (float(min(xs)), float(min(ys)), float(max(xs)), float(max(ys)))



with rasterio.open(href_r) as src_r:

    win = rasterio.windows.from_bounds(*aoi_bounds_s2, transform=src_r.transform)

    win = win.round_offsets().round_lengths()

    pad = 40

    win = rasterio.windows.Window(

        col_off=max(0, win.col_off - pad),

        row_off=max(0, win.row_off - pad),

        width=min(src_r.width - max(0, win.col_off - pad), win.width + 2 * pad),

        height=min(src_r.height - max(0, win.row_off - pad), win.height + 2 * pad),

    )



with rasterio.open(href_r) as src_r, rasterio.open(href_g) as src_g, rasterio.open(href_b) as src_b:

    r, t = _read_window_resampled(src_r, win)

    g, _ = _read_window_resampled(src_g, win)

    b, _ = _read_window_resampled(src_b, win)



rgb = np.dstack([

    _percentile_stretch(r),

    _percentile_stretch(g),

    _percentile_stretch(b),

])



# Reproject crowns to Sentinel CRS for plotting (uses pyproj)

crowns_s2 = crowns.to_crs(s2_crs_str)



fig, ax = plt.subplots(figsize=(10, 10), dpi=160)

ax.imshow(rgb)

ax.set_title(f'Sentinel-2 RGB + crowns | {DATASET_NAME} | {YEAR} | {s2_crs_str} | item {best_item.id}')

ax.axis('off')



# Convert map coords -> pixel coords in the plotted window

inv_t = ~t



for geom in crowns_s2.geometry:

    if geom is None or geom.is_empty:

        continue

    try:

        geoms = [geom] if geom.geom_type == 'Polygon' else list(getattr(geom, 'geoms', []))

        if not geoms:

            continue

        poly = max(geoms, key=lambda p: p.area)

        xs, ys = poly.exterior.xy

        cols_rows = [inv_t * (x, y) for x, y in zip(xs, ys)]

        cols = [cr[0] for cr in cols_rows]

        rows = [cr[1] for cr in cols_rows]

        ax.plot(cols, rows, color='yellow', linewidth=0.8, alpha=0.9)

    except Exception:

        continue



overlay_path = RUN_DIR / 'overlay_s2_rgb_crowns.png'

fig.tight_layout()

fig.savefig(overlay_path)

plt.close(fig)

print('Saved overlay:', overlay_path)

print('Sentinel derived EPSG:', s2_epsg)


Saved overlay: /Users/hbot07/VS Code/Drone-Phenology-Monitoring/output/sat_data/sit_2024_run/overlay_s2_rgb_crowns.png
Sentinel derived EPSG: 32643


In [17]:
# ---- 3) NDVI time series for a few example crowns (saved to disk) ----



def _choose_example_crowns(gdf: gpd.GeoDataFrame, n: int, seed: int = 0) -> gpd.GeoDataFrame:

    rng = random.Random(seed)

    idxs = list(range(len(gdf)))

    rng.shuffle(idxs)

    pick = sorted(idxs[: min(n, len(idxs))])

    out = gdf.iloc[pick].copy()

    out['example_index'] = np.arange(len(out), dtype=int)

    return out





example_crowns = _choose_example_crowns(crowns, N_EXAMPLE_CROWNS, seed=RANDOM_SEED)

examples_path = RUN_DIR / 'example_crowns.geojson'

example_crowns.to_crs('EPSG:4326').to_file(examples_path, driver='GeoJSON')

print('Saved example crowns:', examples_path)





# Keep the number of dates bounded for a first pass (tune as needed)

MAX_DATES = 30

items_df_small = items_df.sort_values(['datetime']).reset_index(drop=True)

if len(items_df_small) > MAX_DATES:

    idxs = np.linspace(0, len(items_df_small) - 1, MAX_DATES).round().astype(int)

    items_df_small = items_df_small.iloc[idxs].reset_index(drop=True)



id_to_item = {it.id: it for it in items}

items_small = [id_to_item[iid] for iid in items_df_small['id'].tolist()]

print('items_used_for_ts:', len(items_small))





def _ndvi_mean_for_polygon(item, poly_wgs84) -> float:

    href_red = _signed_href(item, 'B04')

    href_nir = _signed_href(item, 'B08')



    epsg = _sentinel_epsg_from_item(item, fallback_lat=float(centroid_wgs84.y))

    crs_str = f'EPSG:{int(epsg)}'



    # Reproject polygon to band CRS using geopandas/pyproj

    gtmp = gpd.GeoDataFrame({'id': [1]}, geometry=[poly_wgs84], crs='EPSG:4326').to_crs(crs_str)

    geom = gtmp.geometry.iloc[0]

    if geom is None or geom.is_empty:

        return float('nan')



    with rasterio.open(href_red) as src_red, rasterio.open(href_nir) as src_nir:

        red, _ = rasterio.mask.mask(src_red, [mapping(geom)], crop=True, filled=False)

        nir, _ = rasterio.mask.mask(src_nir, [mapping(geom)], crop=True, filled=False)



        red = red[0].astype('float32')

        nir = nir[0].astype('float32')



        m = np.zeros_like(red, dtype=bool)

        if isinstance(red, np.ma.MaskedArray):

            m |= np.ma.getmaskarray(red)

        if isinstance(nir, np.ma.MaskedArray):

            m |= np.ma.getmaskarray(nir)

        denom = (nir + red)

        m |= ~np.isfinite(denom) | (denom == 0)



        ndvi = (nir - red) / denom

        ndvi = np.where(m, np.nan, ndvi)

        return float(np.nanmean(ndvi))





records = []

ex_wgs84 = example_crowns.to_crs('EPSG:4326')

for it in items_small:

    dt = _item_dt_str(it)

    derived_epsg = _sentinel_epsg_from_item(it, fallback_lat=float(centroid_wgs84.y))

    for _, row in ex_wgs84.iterrows():

        ndvi_mean = _ndvi_mean_for_polygon(it, row.geometry)

        records.append({

            'date': dt,

            'item_id': it.id,

            'cloud_cover': it.properties.get('eo:cloud_cover'),

            'example_index': int(row['example_index']),

            'ndvi_mean': ndvi_mean,

            'derived_epsg': int(derived_epsg),

            'proj:epsg': it.properties.get('proj:epsg'),

        })



ts_df = pd.DataFrame(records)

ts_csv = RUN_DIR / 'ndvi_timeseries_examples.csv'

ts_df.to_csv(ts_csv, index=False)

print('Wrote:', ts_csv)





# Plot combined time series (saved to disk)

ts_df['date'] = pd.to_datetime(ts_df['date'])

fig, ax = plt.subplots(figsize=(10, 4), dpi=160)

for ex_idx, grp in ts_df.groupby('example_index'):

    grp = grp.sort_values('date')

    ax.plot(grp['date'], grp['ndvi_mean'], marker='o', linewidth=1.2, markersize=3, label=f'crown_ex{ex_idx}')

ax.set_title(f'Sentinel-2 NDVI mean over example crowns | {DATASET_NAME} | {YEAR}')

ax.set_ylabel('NDVI')

ax.set_xlabel('date')

ax.grid(True, alpha=0.3)

ax.legend(loc='best', fontsize=8, ncol=2)

fig.tight_layout()

ts_plot = RUN_DIR / 'ndvi_timeseries_examples.png'

fig.savefig(ts_plot)

plt.close(fig)

print('Saved plot:', ts_plot)


Saved example crowns: /Users/hbot07/VS Code/Drone-Phenology-Monitoring/output/sat_data/sit_2024_run/example_crowns.geojson
items_used_for_ts: 30


/var/folders/cx/8v11zwh97cxgdgt_ph13_56r0000gn/T/ipykernel_78956/3740372762.py:121: RuntimeWarning: Mean of empty slice
  return float(np.nanmean(ndvi))
/var/folders/cx/8v11zwh97cxgdgt_ph13_56r0000gn/T/ipykernel_78956/3740372762.py:121: RuntimeWarning: Mean of empty slice
  return float(np.nanmean(ndvi))
/var/folders/cx/8v11zwh97cxgdgt_ph13_56r0000gn/T/ipykernel_78956/3740372762.py:121: RuntimeWarning: Mean of empty slice
  return float(np.nanmean(ndvi))
/var/folders/cx/8v11zwh97cxgdgt_ph13_56r0000gn/T/ipykernel_78956/3740372762.py:121: RuntimeWarning: Mean of empty slice
  return float(np.nanmean(ndvi))
/var/folders/cx/8v11zwh97cxgdgt_ph13_56r0000gn/T/ipykernel_78956/3740372762.py:121: RuntimeWarning: Mean of empty slice
  return float(np.nanmean(ndvi))
/var/folders/cx/8v11zwh97cxgdgt_ph13_56r0000gn/T/ipykernel_78956/3740372762.py:121: RuntimeWarning: Mean of empty slice
  return float(np.nanmean(ndvi))
/var/folders/cx/8v11zwh97cxgdgt_ph13_56r0000gn/T/ipykernel_78956/3740372762.py:121

Wrote: /Users/hbot07/VS Code/Drone-Phenology-Monitoring/output/sat_data/sit_2024_run/ndvi_timeseries_examples.csv
Saved plot: /Users/hbot07/VS Code/Drone-Phenology-Monitoring/output/sat_data/sit_2024_run/ndvi_timeseries_examples.png
